<a href="https://colab.research.google.com/github/liuxiaohu0511/lance-demo/blob/develop/lance_object_storage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pylance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 14.2 MB/s eta 0:00:00


In [2]:
import shutil
import lance
import numpy as np
import pandas as pd
import pyarrow as pa
import duckdb

lance支持aws s3、azure blob store和google cloud storage等对象存储，使用哪个对象存储类型取决于数据集路径的uri方案

对象存储需要额外的配置，这些配置可以通过环境变量或者通过lance.dataset和lance.write_dataset的storage_options参数指定

比如通过环境变量设置全局超时时间
```bash
export TIMEOUT =  60
```

如果只需要为单个数据集设置超时时间，可以通过storage_options参数指定
```python
import lance
ds = lance.dataset("s3://path", storage_options ={"timeout":"60s"})
```

**通用配置**
| key | 描述 |
|------|--------|
| allow_http | 允许非 TLS 连接，即非 HTTPS 连接。默认， False 。 |
| download_retry_count | 重试下载的次数。默认， 3 。此限制适用于 HTTP 请求成功但响应未完全下载的情况，通常是由于违反 request_timeout 。 |
| allow_invalid_certificates | 在 https 连接上跳过证书验证。默认， False 。警告：这是不安全的，仅应用于测试。|
 | connect_timeout	 | 客户端仅连接阶段的超时时间。默认， 5s 。 |
 | request_timeout | 允请求整个过程的超时时间，从连接到响应体完成。默认， 30s 。 |
 | user_agent | 请求中使用的用户代理字符串。 |
 | proxy_url | 用于请求的代理服务器的 URL。默认， None 。 |
  | proxy_ca_certificate | 用于代理连接的 PEM 格式 CA 证书。 |
 | proxy_excludes | 代理绕过主机列表。这是一个以逗号分隔的域名和 IP 掩码列表。任何提供的域的子域都将被绕过。例如， example.com, 192.168.1.0/24 将绕过 https://api.example.com ， https://www.example.com ，以及 192.168.1.0/24 范围内的任何 IP。 |
  | client_max_retries | s3 客户端重试请求的次数。默认， 10 。 |
 | client_retry_timeout | s3 客户端重试请求的超时时间（秒）。默认， 180 。 |

**S3配置**

s3有额外的配置选项，用于配置授权和s3特定功能（比如服务端加密）

AWS 凭证可以设置在环境变量 AWS_ACCESS_KEY_ID 、 AWS_SECRET_ACCESS_KEY 和 AWS_SESSION_TOKEN 中。或者，它们可以作为参数传递给 storage_options 参数

In [3]:
import lance
ds = lance.dataset(
    "s3://bucket/path",
    storage_options={
        "access_key_id": "my-access-key",
        "secret_access_key": "my-secret-key",
        "session_token": "my-session-token",
    }
)

ValueError: Dataset at path path was not found: LanceError(IO): Generic S3 error: Error performing list request: Error performing GET https://s3.us-east-1.amazonaws.com/bucket?list-type=2&prefix=path%2F_versions%2F in 24.21968ms - Server returned non-2xx status code: 403 Forbidden: <?xml version="1.0" encoding="UTF-8"?>
<Error><Code>InvalidAccessKeyId</Code><Message>The AWS Access Key Id you provided does not exist in our records.</Message><AWSAccessKeyId>my-access-key</AWSAccessKeyId><RequestId>7BGDR0BA7EBBXGPK</RequestId><HostId>/xrWKWkBxdYLEpoCmgNjK1mQeAXULG8/KsstEr5BxVonxu8gk/oKOpdOQ68gtv/+XNsT00i+iVXDYj7IsL3QAX4zUsCaBoWI</HostId></Error>, /home/runner/work/lance/lance/rust/lance-io/src/object_store.rs:524:92, /home/runner/work/lance/lance/rust/lance/src/dataset/builder.rs:439:35

**S3兼容配置**
| key | 描述 |
| -------- | -------- |
| aws_region / region	     | 存储桶所在的 AWS 区域。当使用 AWS S3 时，这可以自动检测，但对于兼容 S3 的存储，必须指定。     |
| aws_access_key_id / access_key_id	 | 要使用的 AWS 访问密钥 ID。 |
| aws_secret_access_key / secret_access_key		 | 要使用的 AWS 密钥访问密钥。 |
| aws_session_token / session_token		 | 要使用的 AWS 会话令牌。 |
| aws_endpoint / endpoint		 | 用于 S3 兼容存储的端点。 |
| aws_virtual_hosted_style_request / virtual_hosted_style_request	 | 是否使用虚拟托管样式请求，其中存储桶名称是端点的一部分。与 aws_endpoint 一起使用。默认为 False 。 |
| aws_s3_express / s3_express		 | 是否使用 S3 Express One Zone 端点。默认， False 。更多详情见下文。 |
| aws_server_side_encryption			 | 服务器端加密算法。必须是 "AES256" 、 "aws:kms" 或 "aws:kms:dsse" 之一。默认为 None 。 |
| aws_sse_kms_key_id		 | 用于服务器端加密的 KMS 密钥 ID。如果设置， aws_server_side_encryption 必须是 "aws:kms" 或 "aws:kms:dsse" 。 |
| aws_sse_bucket_key_enabled		 | 是否使用存储桶密钥进行服务器端加密。 |

lance还可以访问兼容s3的存储，例如minio。必须指定region和endpoint。

这也可以通过 AWS_ENDPOINT 和 AWS_DEFAULT_REGION 环境变量来完成。

其他azure blob store和google cloud storage的更多配置参加

https://lancedb.github.io/lance/guide/object_store/#google-cloud-storage-configuration